# Notebook 2: 시나리오 설정 & 실행
## 한국군 방공체계 M&S — 시나리오 1 (포화공격) & 시나리오 5 (노드 파괴)

이 노트북에서는:
- 시나리오 1 (포화공격): 선형 C2 vs Kill Web 단일 실행 비교
- 시나리오 5 (노드 파괴): C2 노드 순차 제거 효과 비교
- 이벤트 로그 & 타임라인 검증

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install simpy mesa networkx matplotlib pandas numpy scipy seaborn --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/K-ADS_Simulation')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from modules.config import *
from modules.model import AirDefenseModel

plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
print('모듈 로드 완료!')

---
## 시나리오 1: 포화공격
### 3파상 (t=0: SRBM 10 + CM 5, t=60s: SRBM 5 + UAS 10, t=120s: CM 5 + UAS 10)

In [ ]:
# 시나리오 1 실행 — 선형 C2
print('=== 시나리오 1: 포화공격 — 선형 C2 ===')
model_s1_linear = AirDefenseModel(
    architecture='linear',
    scenario='scenario_1_saturation',
    seed=42
)
result_s1_linear = model_s1_linear.run_full()

m = result_s1_linear['metrics']
print(f"시뮬레이션 시간: {result_s1_linear['sim_time']}s")
print(f"총 위협: {model_s1_linear.metrics.total_threats}기")
print(f"탐지: {model_s1_linear.metrics.threats_detected}기")
print(f"교전: {model_s1_linear.metrics.threats_engaged}건")
print(f"격추: {model_s1_linear.metrics.total_kills}기")
print(f"누출: {model_s1_linear.metrics.threats_leaked}기")
print(f"\n센서-투-슈터 시간: {m['sensor_to_shooter_time']['mean']:.1f}s (평균)")
print(f"누출률: {m['leaker_rate']:.1f}%")
print(f"교전 성공률: {m['engagement_success_rate']:.1f}%")
print(f"탄약 효율: {m['ammo_efficiency']:.2f} 발/격추")

In [ ]:
# 시나리오 1 실행 — Kill Web
print('=== 시나리오 1: 포화공격 — Kill Web ===')
model_s1_kw = AirDefenseModel(
    architecture='killweb',
    scenario='scenario_1_saturation',
    seed=42
)
result_s1_kw = model_s1_kw.run_full()

m = result_s1_kw['metrics']
print(f"시뮬레이션 시간: {result_s1_kw['sim_time']}s")
print(f"총 위협: {model_s1_kw.metrics.total_threats}기")
print(f"탐지: {model_s1_kw.metrics.threats_detected}기")
print(f"교전: {model_s1_kw.metrics.threats_engaged}건")
print(f"격추: {model_s1_kw.metrics.total_kills}기")
print(f"누출: {model_s1_kw.metrics.threats_leaked}기")
print(f"\n센서-투-슈터 시간: {m['sensor_to_shooter_time']['mean']:.1f}s (평균)")
print(f"누출률: {m['leaker_rate']:.1f}%")
print(f"교전 성공률: {m['engagement_success_rate']:.1f}%")
print(f"탄약 효율: {m['ammo_efficiency']:.2f} 발/격추")

In [ ]:
# 시나리오 1 비교 테이블
comparison = pd.DataFrame({
    '메트릭': [
        '센서-투-슈터 시간 (s)', '누출률 (%)', '교전 성공률 (%)',
        '탄약 효율 (발/격추)', '최대 동시 교전', 'C2 처리량 (건/분)',
    ],
    '선형 C2': [
        f"{result_s1_linear['metrics']['sensor_to_shooter_time']['mean']:.1f}",
        f"{result_s1_linear['metrics']['leaker_rate']:.1f}",
        f"{result_s1_linear['metrics']['engagement_success_rate']:.1f}",
        f"{result_s1_linear['metrics']['ammo_efficiency']:.2f}",
        f"{result_s1_linear['metrics']['max_concurrent_engagements']}",
        f"{result_s1_linear['metrics']['c2_throughput']:.1f}",
    ],
    'Kill Web': [
        f"{result_s1_kw['metrics']['sensor_to_shooter_time']['mean']:.1f}",
        f"{result_s1_kw['metrics']['leaker_rate']:.1f}",
        f"{result_s1_kw['metrics']['engagement_success_rate']:.1f}",
        f"{result_s1_kw['metrics']['ammo_efficiency']:.2f}",
        f"{result_s1_kw['metrics']['max_concurrent_engagements']}",
        f"{result_s1_kw['metrics']['c2_throughput']:.1f}",
    ],
})
print('\n=== 시나리오 1 결과 비교 ===')
print(comparison.to_string(index=False))

---
## 시나리오 5: 노드 파괴
### 선형: MCRC→TOC_PAT→TOC_MSAM 순차 제거 / Kill Web: EOC_1→EOC_2→EOC_3 제거

In [ ]:
# 시나리오 5 실행 — 선형 C2
print('=== 시나리오 5: 노드 파괴 — 선형 C2 ===')
model_s5_linear = AirDefenseModel(
    architecture='linear',
    scenario='scenario_5_node_destruction',
    seed=42
)
result_s5_linear = model_s5_linear.run_full()

m = result_s5_linear['metrics']
print(f"시뮬레이션 시간: {result_s5_linear['sim_time']}s")
print(f"격추: {model_s5_linear.metrics.total_kills}기")
print(f"누출: {model_s5_linear.metrics.threats_leaked}기")
print(f"누출률: {m['leaker_rate']:.1f}%")
print(f"체계 회복탄력성: {m['system_resilience']:.1f}%")
print(f"노드 손실 후 복구시간: {m['node_loss_recovery_time']:.1f}s")

In [ ]:
# 시나리오 5 실행 — Kill Web
print('=== 시나리오 5: 노드 파괴 — Kill Web ===')
model_s5_kw = AirDefenseModel(
    architecture='killweb',
    scenario='scenario_5_node_destruction',
    seed=42
)
result_s5_kw = model_s5_kw.run_full()

m = result_s5_kw['metrics']
print(f"시뮬레이션 시간: {result_s5_kw['sim_time']}s")
print(f"격추: {model_s5_kw.metrics.total_kills}기")
print(f"누출: {model_s5_kw.metrics.threats_leaked}기")
print(f"누출률: {m['leaker_rate']:.1f}%")
print(f"체계 회복탄력성: {m['system_resilience']:.1f}%")
print(f"노드 손실 후 복구시간: {m['node_loss_recovery_time']:.1f}s")

In [ ]:
# 시나리오 5 비교 시각화 — 노드 파괴 전후 성능 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. 누출률 비교
scenarios = ['S1 포화공격', 'S5 노드파괴']
linear_leaker = [result_s1_linear['metrics']['leaker_rate'],
                 result_s5_linear['metrics']['leaker_rate']]
kw_leaker = [result_s1_kw['metrics']['leaker_rate'],
             result_s5_kw['metrics']['leaker_rate']]

x = np.arange(len(scenarios))
w = 0.35
axes[0].bar(x - w/2, linear_leaker, w, label='선형 C2', color='#FF7043')
axes[0].bar(x + w/2, kw_leaker, w, label='Kill Web', color='#42A5F5')
axes[0].set_ylabel('누출률 (%)')
axes[0].set_title('누출률 비교')
axes[0].set_xticks(x)
axes[0].set_xticklabels(scenarios)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# 2. 센서-투-슈터 시간
linear_s2s = [result_s1_linear['metrics']['sensor_to_shooter_time']['mean'],
              result_s5_linear['metrics']['sensor_to_shooter_time']['mean']]
kw_s2s = [result_s1_kw['metrics']['sensor_to_shooter_time']['mean'],
          result_s5_kw['metrics']['sensor_to_shooter_time']['mean']]

axes[1].bar(x - w/2, linear_s2s, w, label='선형 C2', color='#FF7043')
axes[1].bar(x + w/2, kw_s2s, w, label='Kill Web', color='#42A5F5')
axes[1].set_ylabel('시간 (초)')
axes[1].set_title('센서-투-슈터 시간 비교')
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenarios)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# 3. 체계 회복탄력성 (시나리오 5)
resilience = [
    result_s5_linear['metrics']['system_resilience'],
    result_s5_kw['metrics']['system_resilience'],
]
colors = ['#FF7043', '#42A5F5']
axes[2].bar(['선형 C2', 'Kill Web'], resilience, color=colors)
axes[2].set_ylabel('회복탄력성 (%)')
axes[2].set_title('시나리오 5: 체계 회복탄력성')
axes[2].grid(axis='y', alpha=0.3)

fig.suptitle('시나리오 1 & 5 결과 비교: 선형 C2 vs Kill Web', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/scenario_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 이벤트 로그 분석

In [ ]:
# 이벤트 로그를 DataFrame으로 변환
def events_to_df(event_log):
    return pd.DataFrame(event_log)

df_linear = events_to_df(result_s1_linear['event_log'])
df_kw = events_to_df(result_s1_kw['event_log'])

print('=== 선형 C2 이벤트 분포 ===')
if not df_linear.empty:
    print(df_linear['event'].value_counts())
else:
    print('이벤트 없음')

print('\n=== Kill Web 이벤트 분포 ===')
if not df_kw.empty:
    print(df_kw['event'].value_counts())
else:
    print('이벤트 없음')